# Dipole Analysis in Fink/LSST DIA Alerts

This notebook investigates whether dipole-flagged DIA sources (`r:isDipole == True`)
can explain why Gaia-stable stars appear in Fink alerts.

**Hypothesis:** PSF-subtraction residuals (dipoles) may create spurious detections
that trigger the Rubin AP pipeline, causing intrinsically stable stars to appear
as variable in Fink broker data.

**Analysis structure:**
1. Load `*_src.parquet` files from `data_FINK_BLOCK_LC_01/`
2. For each stellar category and photometric band:
   - Histogram of dipole-source count vs. `scienceFlux` (AB mag)
   - Histogram of dipole fraction (N_dipole / N_total) vs. `scienceFlux` (AB mag)
   - Same two histograms vs. `psfFlux` (AB mag)
3. Additional diagnostics: dipole fraction vs. `templateFlux`, band, per-object rate
4. Comparative summary heatmap across categories

**Categories analysed:**
- `gaia_star_stable_hq` — high-quality Gaia stable stars (calibration targets)
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable but no photometric solution
- `gaia_star_variable` — Gaia variable stars (control group)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-13  
**Subject:** Rubin LSST — Fink alert broker — dipole trigger hypothesis

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input from notebook 01
DIR_FIGS = f"figs_{NB_TAG}_11"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Source categories to analyse ──────────────────────────────────────────────
# key   = parquet filename stem (before _src.parquet)
# label = short human-readable label for plots
# color = matplotlib colour for this category
CATEGORIES = {
    "gaia_star_stable_hq": {"label": "Gaia stable HQ", "color": "steelblue"},
    "gaia_nophotgstar_stable_unknown_parallax": {"label": "Gaia stable no-phot", "color": "seagreen"},
    "gaia_star_variable": {"label": "Gaia variable", "color": "firebrick"},
}

# ── AB magnitude calibration constant ────────────────────────────────────────
# m_AB = -2.5 * log10(f_nJy / 3631e9)
AB_FLUX_ZERO = 3631e9  # nJy

# ── Histogram binning ────────────────────────────────────────────────────────
# AB magnitude bins — covers faint end of LSST (~27) down to bright limit (~14)
MAG_BINS = np.arange(14, 28.5, 0.5)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f / AB_FLUX_ZERO), np.nan)


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


def dipole_fraction_per_bin(mag_all, mag_dip, bins):
    """
    Compute per-bin counts and dipole fraction with 1-sigma Wilson error.

    Parameters
    ----------
    mag_all : array-like  -- AB magnitudes of ALL sources
    mag_dip : array-like  -- AB magnitudes of dipole-flagged sources
    bins    : array-like  -- magnitude bin edges

    Returns
    -------
    bin_centres, n_total, n_dipole, frac_dipole, frac_err
    """
    n_total, _ = np.histogram(mag_all, bins=bins)
    n_dipole, _ = np.histogram(mag_dip, bins=bins)

    with np.errstate(invalid="ignore", divide="ignore"):
        frac = np.where(n_total > 0, n_dipole / n_total, np.nan)
        p = np.where(n_total > 0, n_dipole / n_total, np.nan)
        n = np.where(n_total > 0, n_total.astype(float), np.nan)
        frac_err = np.where(n_total > 0, np.sqrt(p * (1 - p) / n), np.nan)

    bin_centres = 0.5 * (bins[:-1] + bins[1:])
    return bin_centres, n_total, n_dipole, frac, frac_err


print("Utility functions defined.")

## 3. Load parquet data

We load the `*_src.parquet` files (DIA detections = alert sources).  
The `*_fp.parquet` files (forced photometry) do **not** carry `r:isDipole` and are not needed here.

**Note on `scienceFlux`:** Fink alert packets do not always expose `r:scienceFlux`
directly in the parquet output.  When absent, we fall back to `|psfFlux|` as a
proxy (the absolute difference-image flux), with an info message.

In [ ]:
data = {}  # data[cat] = cleaned DataFrame for that category

for cat in CATEGORIES:
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Minimal column guard ──────────────────────────────────────────────
    required = ["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr", "r:band"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[WARN] {cat}: missing columns {missing} — skipped")
        continue

    df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

    # ── Normalise isDipole flag ───────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: no r:isDipole column — all set to False")
        df["is_dipole"] = False

    # ── psfFlux AB magnitude  (difference-image flux, can be negative) ────
    df["mag_psf"] = flux_nJy_to_mag_AB(df["r:psfFlux"].values)
    df["mag_psf_abs"] = flux_nJy_to_mag_AB(np.abs(df["r:psfFlux"].values))

    # ── scienceFlux AB magnitude  (pre-subtraction brightness) ───────────
    if "r:scienceFlux" in df.columns:
        df["mag_science"] = flux_nJy_to_mag_AB(df["r:scienceFlux"].values)
    else:
        print(f"[INFO] {cat}: r:scienceFlux absent — using |psfFlux| as proxy for mag_science")
        df["mag_science"] = df["mag_psf_abs"]

    # ── templateFlux AB magnitude  (coadd brightness) ────────────────────
    if "r:templateFlux" in df.columns:
        df["mag_template"] = flux_nJy_to_mag_AB(df["r:templateFlux"].values)
    else:
        df["mag_template"] = np.nan

    data[cat] = df
    lbl = CATEGORIES[cat]["label"]
    n_tot = len(df)
    n_dip = int(df["is_dipole"].sum())
    print(f"  {lbl:35s}  {n_tot:6d} rows  {n_dip:5d} dipoles  ({100 * n_dip / n_tot:.1f}%)")

CATS_OK = list(data.keys())
print(f"\nCategories loaded: {CATS_OK}")

## 4. Quick inspection

In [ ]:
# Available columns
if CATS_OK:
    cat0 = CATS_OK[0]
    print(f"Columns in {cat0}_src.parquet ({len(data[cat0].columns)} total):")
    for c in sorted(data[cat0].columns):
        print(f"  {c}")

In [ ]:
# Global dipole statistics per category
rows = []
for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    n = len(df)
    n_d = int(df["is_dipole"].sum())
    frac = n_d / n if n > 0 else np.nan
    n_obj = df["diaObjectId"].nunique() if "diaObjectId" in df.columns else np.nan
    rows.append(
        {
            "category": lbl,
            "n_objects": n_obj,
            "n_alerts": n,
            "n_dipoles": n_d,
            "dipole_frac_%": round(100 * frac, 2),
        }
    )

print("Global dipole statistics per category:")
display(pd.DataFrame(rows))

## 5. Dipole histograms vs. scienceFlux (AB mag) — per category × band

For each stellar category, a 2-row × 6-column figure:
- **Row 0:** absolute count of all sources (transparent fill) and dipole sources (red) per magnitude bin  
- **Row 1:** dipole fraction N_dip / N_tot per bin with 1σ Wilson error bars

The science-image flux measures the star's true brightness before template subtraction —
the most physically meaningful axis to test whether bright stars produce more dipoles.

In [ ]:
def plot_dipole_histograms_per_band(
    df, cat_label, cat_color, mag_col, xlabel_base, bins=MAG_BINS, savename=None
):
    """
    2-row × 6-band figure showing dipole counts (top) and fraction (bottom)
    as a function of AB magnitude for one stellar category.

    Parameters
    ----------
    df          : DataFrame  -- src data for one category
    cat_label   : str        -- category name for suptitle
    cat_color   : str        -- category colour (used for band title)
    mag_col     : str        -- column name with precomputed AB magnitude values
    xlabel_base : str        -- x-axis label prefix (e.g. 'scienceFlux')
    bins        : array      -- magnitude bin edges
    savename    : str|None   -- if given, save PDF + PNG to DIR_FIGS
    """
    fig, axes = plt.subplots(
        2, len(BANDS), figsize=(3.0 * len(BANDS), 6), sharex=True, sharey="row", squeeze=False
    )
    fig.suptitle(f"{cat_label} — dipoles vs {xlabel_base}", fontsize=11, fontweight="bold", y=1.01)

    for col_idx, band in enumerate(BANDS):
        bcolor = BAND_COLORS[band]
        df_b = df[df["r:band"] == band]
        df_b_d = df_b[df_b["is_dipole"]]

        mag_all = df_b[mag_col].dropna().values
        mag_dip = df_b_d[mag_col].dropna().values

        bin_cen, n_tot, n_dip, frac, frac_err = dipole_fraction_per_bin(mag_all, mag_dip, bins)
        bw = np.diff(bins)

        # ── Row 0: absolute counts ────────────────────────────────────────
        ax0 = axes[0][col_idx]
        ax0.bar(bin_cen, n_tot, width=bw, alpha=0.30, color=bcolor, label="all", align="center")
        ax0.bar(bin_cen, n_dip, width=bw, alpha=0.80, color="tomato", label="dipole", align="center")
        ax0.set_title(f"band {band}", color=bcolor, fontweight="bold")
        if col_idx == 0:
            ax0.set_ylabel("N sources")
        ax0.legend(fontsize=7, loc="upper left")
        # Use log scale only if there are data; guard against empty bands
        if n_tot.sum() > 0:
            ax0.set_yscale("log")
            ax0.set_ylim(bottom=0.5)

        # ── Row 1: fraction ───────────────────────────────────────────────
        ax1 = axes[1][col_idx]
        valid = np.isfinite(frac) & (n_tot > 0)
        if valid.any():
            ax1.bar(
                bin_cen[valid],
                frac[valid],
                width=bw[valid],
                alpha=0.75,
                color="tomato",
                align="center",
                yerr=frac_err[valid],
                capsize=2,
                error_kw={"linewidth": 0.8, "ecolor": "darkred"},
            )
        ax1.set_xlabel(f"{xlabel_base} (AB mag)")
        if col_idx == 0:
            ax1.set_ylabel("N_dip / N_tot")
        ax1.set_ylim(0, 1.05)
        ax1.axhline(0.5, ls="--", color="grey", lw=0.8, alpha=0.6, label="50%")
        ax1.axhline(0.1, ls=":", color="grey", lw=0.8, alpha=0.6, label="10%")
        if col_idx == 0:
            ax1.legend(fontsize=6, loc="upper left")

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_dipole_histograms_per_band() defined.")

In [ ]:
for cat in CATS_OK:
    lbl = CATEGORIES[cat]["label"]
    color = CATEGORIES[cat]["color"]
    safe = cat.replace("/", "_")
    print(f"\n=== {lbl} — scienceFlux ===")
    plot_dipole_histograms_per_band(
        data[cat],
        lbl,
        color,
        mag_col="mag_science",
        xlabel_base="scienceFlux",
        savename=f"11a_dipoles_scienceFlux_{safe}",
    )

## 6. Dipole histograms vs. |psfFlux| (AB mag) — per category × band

The PSF flux is the flux measured in the **difference image** (science − template).
For stable stars it should scatter around zero; for real variables it carries a signal.
We use the absolute value to convert to AB magnitudes.
Dipoles piling up at faint |psfFlux| would indicate noise-driven artefacts,
while dipoles at bright |psfFlux| suggest large PSF-matching residuals.

In [ ]:
for cat in CATS_OK:
    lbl = CATEGORIES[cat]["label"]
    color = CATEGORIES[cat]["color"]
    safe = cat.replace("/", "_")
    print(f"\n=== {lbl} — |psfFlux| ===")
    plot_dipole_histograms_per_band(
        data[cat],
        lbl,
        color,
        mag_col="mag_psf_abs",
        xlabel_base="|psfFlux|",
        savename=f"11b_dipoles_psfFlux_{safe}",
    )

## 7. Comparison: dipole fraction vs. scienceFlux — all categories overlaid

One row of 6 axes (one per band).  Each axis shows the dipole fraction curve for
all three categories with error bars, allowing direct visual comparison.

In [ ]:
def plot_dipole_fraction_comparison(
    data_dict, categories, mag_col, xlabel_base, bins=MAG_BINS, savename=None
):
    """
    1-row × 6-band figure: dipole fraction vs AB magnitude for all categories overlaid.
    """
    fig, axes = plt.subplots(1, len(BANDS), figsize=(3.2 * len(BANDS), 4), sharey=True, squeeze=False)
    fig.suptitle(f"Dipole fraction vs {xlabel_base} — all categories", fontsize=11, fontweight="bold", y=1.02)

    for col_idx, band in enumerate(BANDS):
        ax = axes[0][col_idx]
        ax.set_title(f"band {band}", color=BAND_COLORS[band], fontweight="bold")

        for cat in categories:
            if cat not in data_dict:
                continue
            df = data_dict[cat]
            lbl = CATEGORIES[cat]["label"]
            ccolor = CATEGORIES[cat]["color"]

            df_b = df[df["r:band"] == band]
            df_b_d = df_b[df_b["is_dipole"]]

            bin_cen, n_tot, n_dip, frac, frac_err = dipole_fraction_per_bin(
                df_b[mag_col].dropna().values, df_b_d[mag_col].dropna().values, bins
            )

            valid = np.isfinite(frac) & (n_tot > 0)
            if valid.any():
                ax.errorbar(
                    bin_cen[valid],
                    frac[valid],
                    yerr=frac_err[valid],
                    fmt="o-",
                    ms=4,
                    lw=1.2,
                    capsize=2,
                    color=ccolor,
                    label=lbl,
                    alpha=0.85,
                )

        ax.set_xlabel(f"{xlabel_base} (AB mag)")
        ax.set_ylim(0, 1.05)
        ax.axhline(0.5, ls="--", color="grey", lw=0.7, alpha=0.5)
        ax.axhline(0.1, ls=":", color="grey", lw=0.7, alpha=0.5)
        if col_idx == 0:
            ax.set_ylabel("N_dip / N_tot")
            ax.legend(fontsize=7)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_dipole_fraction_comparison() defined.")

In [ ]:
plot_dipole_fraction_comparison(
    data,
    CATS_OK,
    mag_col="mag_science",
    xlabel_base="scienceFlux",
    savename="11c_dipole_frac_compare_scienceFlux",
)

In [ ]:
plot_dipole_fraction_comparison(
    data, CATS_OK, mag_col="mag_psf_abs", xlabel_base="|psfFlux|", savename="11d_dipole_frac_compare_psfFlux"
)

## 8. Comparison: dipole count vs. scienceFlux — all categories overlaid

Absolute dipole counts per magnitude bin as step histograms, overlaid for all categories.
This reveals the *incidence* of dipoles independently of the total number of alerts
in each category.

In [ ]:
def plot_dipole_count_comparison(data_dict, categories, mag_col, xlabel_base, bins=MAG_BINS, savename=None):
    """1-row × 6-band: dipole *counts* per bin overlaid for all categories."""
    fig, axes = plt.subplots(1, len(BANDS), figsize=(3.2 * len(BANDS), 4), sharey=False, squeeze=False)
    fig.suptitle(f"Dipole count vs {xlabel_base} — all categories", fontsize=11, fontweight="bold", y=1.02)

    for col_idx, band in enumerate(BANDS):
        ax = axes[0][col_idx]
        ax.set_title(f"band {band}", color=BAND_COLORS[band], fontweight="bold")

        for cat in categories:
            if cat not in data_dict:
                continue
            df = data_dict[cat]
            lbl = CATEGORIES[cat]["label"]
            ccolor = CATEGORIES[cat]["color"]

            mag_dip = df[(df["r:band"] == band) & df["is_dipole"]][mag_col].dropna().values
            n_dip, _ = np.histogram(mag_dip, bins=bins)

            # Step histogram: replicate last value for closing edge
            ax.step(bins[:-1], n_dip, where="post", lw=1.5, color=ccolor, label=lbl, alpha=0.85)

        ax.set_xlabel(f"{xlabel_base} (AB mag)")
        if col_idx == 0:
            ax.set_ylabel("N dipoles")
            ax.legend(fontsize=7)
        if ax.get_ylim()[1] > 1:
            ax.set_yscale("log")
            ax.set_ylim(bottom=0.5)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_dipole_count_comparison() defined.")

In [ ]:
plot_dipole_count_comparison(
    data,
    CATS_OK,
    mag_col="mag_science",
    xlabel_base="scienceFlux",
    savename="11e_dipole_count_compare_scienceFlux",
)

plot_dipole_count_comparison(
    data, CATS_OK, mag_col="mag_psf_abs", xlabel_base="|psfFlux|", savename="11f_dipole_count_compare_psfFlux"
)

## 9. Additional diagnostic: dipole fraction vs. templateFlux

The template flux is the coadded reference brightness.  A brighter template may
generate stronger PSF-matching residuals and therefore more dipoles.  This section
tests that hypothesis (executed only if `r:templateFlux` is present).

In [ ]:
any_template = any(
    "r:templateFlux" in data[cat].columns and data[cat]["mag_template"].notna().any() for cat in CATS_OK
)

if any_template:
    print("templateFlux available — plotting dipole fraction vs templateFlux")
    plot_dipole_fraction_comparison(
        data,
        CATS_OK,
        mag_col="mag_template",
        xlabel_base="templateFlux",
        savename="11g_dipole_frac_compare_templateFlux",
    )
else:
    print("r:templateFlux not present or all NaN in loaded data — section skipped.")

## 10. Dipole fraction per photometric band — global view

Bar chart of the **global** dipole fraction (all magnitudes combined) per band,
for each category.  PSF size, template depth, and sky background all vary with
band; this plot reveals whether some bands are systematically more affected.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(BANDS))
width = 0.22
n_cats = len(CATS_OK)
offsets = np.linspace(-(n_cats - 1) * width / 2, (n_cats - 1) * width / 2, n_cats)

for k, (cat, offset) in enumerate(zip(CATS_OK, offsets)):
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    color = CATEGORIES[cat]["color"]

    fracs, errs = [], []
    for band in BANDS:
        df_b = df[df["r:band"] == band]
        n_tot = len(df_b)
        n_dip = int(df_b["is_dipole"].sum())
        if n_tot > 0:
            p = n_dip / n_tot
            fracs.append(p)
            errs.append(np.sqrt(p * (1 - p) / n_tot))
        else:
            fracs.append(np.nan)
            errs.append(np.nan)

    ax.bar(
        x + offset,
        fracs,
        width=width * 0.9,
        yerr=errs,
        color=color,
        alpha=0.80,
        label=lbl,
        capsize=3,
        error_kw={"linewidth": 0.9},
    )

ax.set_xticks(x)
ax.set_xticklabels(BANDS, fontsize=10)
ax.set_xlabel("Photometric band")
ax.set_ylabel("Dipole fraction  N_dip / N_tot")
ax.set_title("Global dipole fraction per band (all magnitudes)", fontweight="bold")
ax.legend(fontsize=8)
ax.set_ylim(0, min(1.0, ax.get_ylim()[1] * 1.20))

plt.tight_layout()
savefig("11h_dipole_frac_per_band")
plt.show()

## 11. Per-object dipole rate distribution

Distribution of the **per-object dipole fraction** (fraction of a given star's alerts
flagged as dipoles).  Distinguishes two regimes:
- fraction ≈ 0 → star never produces dipoles (PSF match OK)
- fraction ≈ 1 → star consistently produces dipoles (structural problem)

A spike at 1.0 with many objects points to a systematic rather than statistical origin.

In [ ]:
if "diaObjectId" not in data[CATS_OK[0]].columns:
    print("diaObjectId column not found — section skipped.")
else:
    fig, axes = plt.subplots(1, len(CATS_OK), figsize=(5 * len(CATS_OK), 4), sharey=False, squeeze=False)
    fig.suptitle("Per-object dipole fraction distribution", fontsize=11, fontweight="bold", y=1.02)

    for col_idx, cat in enumerate(CATS_OK):
        df = data[cat]
        lbl = CATEGORIES[cat]["label"]
        color = CATEGORIES[cat]["color"]
        ax = axes[0][col_idx]

        obj_stats = (
            df.groupby("diaObjectId")
            .agg(n_alerts=("is_dipole", "count"), n_dipoles=("is_dipole", "sum"))
            .assign(dip_frac=lambda x: x["n_dipoles"] / x["n_alerts"])
            .reset_index()
        )

        n_all_dip = (obj_stats["dip_frac"] == 1.0).sum()
        median_frac = obj_stats["dip_frac"].median()

        ax.hist(
            obj_stats["dip_frac"],
            bins=np.linspace(0, 1, 21),
            color=color,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.4,
        )
        ax.axvline(median_frac, color="k", lw=1.2, ls="--", label=f"median = {median_frac:.2f}")
        ax.set_title(f"{lbl}\nN_obj={len(obj_stats)}, all-dip objects: {n_all_dip}", fontsize=8)
        ax.set_xlabel("Dipole fraction per object")
        if col_idx == 0:
            ax.set_ylabel("N objects")
        ax.legend(fontsize=7)

    plt.tight_layout()
    savefig("11i_per_object_dipole_fraction")
    plt.show()

## 12. Top dipole-dominated objects

For each category: the 15 objects with the highest dipole fraction,
ranked by `dip_frac` (descending), with their mean flux and bands covered.
These are the primary candidates for Butler difference-image stamp inspection.

In [ ]:
if "diaObjectId" not in data[CATS_OK[0]].columns:
    print("diaObjectId column not found — section skipped.")
else:
    for cat in CATS_OK:
        df = data[cat]
        lbl = CATEGORIES[cat]["label"]

        obj_stats = (
            df.groupby("diaObjectId")
            .agg(
                n_alerts=("is_dipole", "count"),
                n_dipoles=("is_dipole", "sum"),
                mean_psf_nJy=("r:psfFlux", "mean"),
                band_list=("r:band", lambda x: ",".join(sorted(x.unique()))),
            )
            .assign(dip_frac=lambda x: x["n_dipoles"] / x["n_alerts"])
            .reset_index()
        )
        obj_stats["mean_mag_AB"] = flux_nJy_to_mag_AB(np.abs(obj_stats["mean_psf_nJy"].values))

        top = (
            obj_stats[obj_stats["n_dipoles"] > 0]
            .sort_values("dip_frac", ascending=False)
            .head(15)
            .reset_index(drop=True)
        )

        print(f"\n{'=' * 65}")
        print(f"  {lbl} — top 15 objects by dipole fraction")
        print(f"{'=' * 65}")
        display(
            top[
                ["diaObjectId", "n_alerts", "n_dipoles", "dip_frac", "mean_mag_AB", "band_list"]
            ].style.format({"dip_frac": "{:.2f}", "mean_mag_AB": "{:.2f}"})
        )

## 13. Comparative summary table

Compact `N_dip (frac%)` table: rows = category, columns = band + ALL.

In [ ]:
rows_summary = []
for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    row = {"Category": lbl}
    n_tot_all = len(df)
    n_dip_all = int(df["is_dipole"].sum())
    row["ALL"] = f"{n_dip_all} ({100 * n_dip_all / n_tot_all:.1f}%)" if n_tot_all > 0 else "—"

    for band in BANDS:
        df_b = df[df["r:band"] == band]
        n_tot = len(df_b)
        n_dip = int(df_b["is_dipole"].sum())
        row[band] = f"{n_dip} ({100 * n_dip / n_tot:.1f}%)" if n_tot > 0 else "—"
    rows_summary.append(row)

df_comp = pd.DataFrame(rows_summary).set_index("Category")
print("Comparative dipole summary  [N_dip (fraction%)] per category × band:")
display(df_comp)

In [ ]:
# Numerical version for export or downstream analysis
rows_num = []
for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    for band in ["all"] + BANDS:
        df_b = df if band == "all" else df[df["r:band"] == band]
        n_tot = len(df_b)
        n_dip = int(df_b["is_dipole"].sum())
        frac = n_dip / n_tot if n_tot > 0 else np.nan
        rows_num.append(
            {
                "category": lbl,
                "band": band,
                "n_total": n_tot,
                "n_dipole": n_dip,
                "dipole_frac": round(frac, 4) if np.isfinite(frac) else np.nan,
            }
        )

df_num = pd.DataFrame(rows_num)
print("\nNumerical summary (all bands + per-band):")
display(df_num)

## 14. Comparative heatmap

Heatmap of dipole fraction (%) with annotated cell values:
rows = category, columns = photometric band.

In [ ]:
pivot = (
    df_num[df_num["band"].isin(BANDS)]
    .pivot(index="category", columns="band", values="dipole_frac")
    .reindex(columns=BANDS)  # enforce ugrizy column order
)

fig, ax = plt.subplots(figsize=(9, 2.0 + 0.8 * len(CATS_OK)))
im = ax.imshow(pivot.values * 100, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)

ax.set_xticks(range(len(BANDS)))
ax.set_xticklabels(BANDS, fontsize=10)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title("Dipole fraction (%) per category × band", fontweight="bold", pad=10)

for i in range(len(pivot)):
    for j in range(len(BANDS)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            tc = "white" if val * 100 > 55 else "black"
            ax.text(j, i, f"{100 * val:.1f}%", ha="center", va="center", fontsize=9, color=tc)

cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("Dipole fraction (%)")

plt.tight_layout()
savefig("11j_dipole_frac_heatmap")
plt.show()

## 15. Conclusions

Key questions this notebook answers:

1. **Are dipoles more frequent for stable stars than for variable stars?**  
   Compare the `Gaia stable HQ` and `Gaia stable no-phot` rows against `Gaia variable`
   in the heatmap (Section 14) and in the per-band bar chart (Section 10).

2. **At which brightness do dipoles preferentially occur?**  
   The fraction histograms (Sections 5–7) show whether the dipole rate peaks at
   the bright end (PSF-matching residuals on bright templates) or at the faint end
   (noise-driven artefacts at the detection threshold).

3. **Are dipoles concentrated on a few objects or spread across many?**  
   The per-object fraction distribution (Section 11) answers this.  A spike at
   frac = 1.0 means a handful of persistently mismatched stars; a flat distribution
   means a global pipeline issue.

4. **Which photometric bands are most affected?**  
   Read from Section 10 (bar chart) and Section 14 (heatmap).

> **Recommended next step:** take the `diaObjectId` + `(visitId, detector)` pairs
> of the top dipole-dominated objects (Section 12) and inspect the Butler difference
> image stamps to confirm the dipole morphology visually — and to check whether the
> positive and negative lobes align with the star centroid displacement between
> science and template.

In [ ]:
print("Notebook 11 — dipole analysis — complete.")